# 0.1 + 0.2 != 0.3

Written by:

* Philip Ilten (University of Cincinnati)

In this notebook we explore some common issues that arise when trying to implement math with computers.

## Requirements

Before getting started, we need to import a few modules. We will use `sys` and `platform` to determine system properties. We will use `fractions` and `decimal` for high-precision calculations. Finally, we will use `matplotlib` for plotting.

In [ ]:
# Import the `sys` and `platform` modules.
import sys, platform

# Import the `fraction` and `decimal` modules.
import fractions, decimal

# Import `matplotlib`.
import matplotlib.pyplot as plt

## Introduction

When doing calculations with computers, we oftentimes take for granted that the computer is doing what we think it should be doing. There's a reason for this, most of the time the computer *does* do what we think it should be doing. But, sometimes it doesn't and this behaviour can depend upon the setup used to produce the calculation: computer hardware (CPU, GPU, memory, *etc.*), operating system, compiler, programming language, and algorithm.

Let's start off with a simple example. First, we are going to break some things, so it would be good to know what our setup is.

In [ ]:
# What kind of CPU are we running on?
# The exclamation mark at the front in iPython runs a shell command
# outside of Python.
!lscpu
print("-" * 80)

# What about our operating system?
!cat /etc/os-release
print("-" * 80)

# What interpreter am I using?
print(platform.python_implementation())
print("-" * 80)

# What version of Python are we using?
!python --version
print("-" * 80)

This is a lot of information! In the end, we won't use most of it, but it is good to know that we are using `Python 3` on a 64-bit CPU. We will use this information a little bit later.

So, let's try an example. Will the following cell evaluate to `True` or `False`?

In [ ]:
1 + 2 == 3

Ok, so far, so good. Hopefully this is what we expect. Let's try this again!

In [ ]:
0.1 + 0.2 == 0.3

Well, that was unexpected. What's going on?

## Numerical Precision

The short answer is numerical precision. What about the slightly longer answer? First, what happens if we write out $1/3$ in decimal form?
$$
1/3 = 0.3\bar{3}
$$
We end up with our last digit $3$ repeating. We can't write $1/3$ in decimal form with a finite number of digits.

Now, let's do the same thing for $1/10$. But here's the catch, computers don't work in a base $10$ math system (decimal), but rather base $2$ (binary). So, we need to write $1/10$ in binary. How do we do that for a fractional decimal?

1. Take the fractional part of our decimal number, in this case just $0.1$ and multiply this by $2$.
$$
0.1\times2 = \color{red}{0}.2
$$
2. Take the digit to the left of the decimal point (highlighted in red) as our first binary digit.
$$
1/10 = 0.1 = 0.\color{red}{0}\ldots_2
$$
3. Subtract this digit from $0.2$. If $0$, we're done.
$$
0.2 - \color{red}{0} = 0.2
$$
   Looks like we're not done ...
4. If not $0$, return to step 1 but using this difference.
$$
\begin{aligned}
0.2\times2 &= \color{orange}{0}.4 \\
1/10 = 0.1 &= 0.0\color{orange}{0}\ldots_2 \\
0.4 - \color{orange}{0} &= 0.4 \\
0.4\times2 &= \color{yellow}{0}.8 \\
1/10 = 0.1 &= 0.00\color{yellow}{0}\ldots_2 \\
0.8 - \color{yellow}{0} &= 0.8 \\
0.8\times2 &= \color{green}{1}.6 \\
1/10 = 0.1 &= 0.000\color{green}{1}\ldots_2 \\
1.6 - \color{green}{1} &= 0.6 \\
0.6\times2 &= \color{blue}{1}.2 \\
1/10 = 0.1 &= 0.0001\color{blue}{1}\ldots_2 \\
1.2 - \color{blue}{1} &= 0.2 \\
0.2\times2 &= \color{purple}{0}.4 \\
1/10 = 0.1 &= 0.00011\color{purple}{0}\ldots_2 \\
\end{aligned}
$$

Wait, we're just back to $0.2$, this whole process will continue on forever! It turns out that just like $1/3$ in decimal, $1/10$ in binary cannot be expressed with a finite number of digits.
$$
1/10 = 0.0\overline{0011}_2
$$
If we want to store this number to full precision with a computer, we would need an infinite number of bits! How many do we have?

In [ ]:
# `getsizeof` of returns the number of bytes for an object.
total = sys.getsizeof(0.1)
# But, every CPython object has an overhead of 16 bytes.
overhead = 16
raw = total - overhead
# Finally, we need to go from bytes to bits.
# 8 bits = 1 byte
raw * 8

Well, that's certainly not infinite! Actually, we might have expected this given the info of the CPU we are running on, since it did tell us it was `64`-bit. Let's see the practical effect of this truncation.

In [ ]:
0.1 + 0.2

At the 17th digit, we run into a problem. Do we expect this, given what we know about representing these fractions in binary? Let us explore a little example where we see what happens when we perform fixed point addition. This is not quite what `Python` is doing with `0.1 + 0.2`, but very similar.

In [ ]:
# First, we write a function that implements our binary conversion above.
def dec_to_bin(num, den, prec=10):
    """
    Convert the decimal fraction with numerator `num` and denominator `den`
    into a binary fraction with precision `prec` using the algorithm above. The
    result is a string to avoid numerical representation issues.
    """
    binary = ""
    # The `Fraction` class stores the numerator and denominator both as
    # integers.
    diff = fractions.Fraction(num, den)
    while len(binary) < prec and diff != 0:
        # Step 1, multiply by 2.
        diff *= 2
        # Step 2, find the charateristic (integer to the left of the
        # decimal point), and append to the binary digits.
        char = int(diff)
        binary += str(char)
        # Step 3, take the difference.
        diff -= char
    # Return the result (pad with zeros to the requested precision).
    return binary.ljust(prec, "0")

In [ ]:
# Second, we write a function that can add our two binary strings together.
def add_bin(a, b):
    """
    Add two strings representing binary fractions together, `a + b`. This
    only returns the fractional component of the sum.
    """
    # Make sure both binary strings are the same length.
    n = max(len(a), len(b))
    a, b = a.ljust(n, "0"), b.ljust(n, "0")
    # This is the sum of the two binary fraction strings.
    total = ""
    # This is what we carry over when adding each set of digits.
    quotient = 0
    # Loop over the bits backward.
    for aBit, bBit in zip(reversed(a), reversed(b)):
        # Add the bits together.
        quotient, remainder = divmod(quotient + int(aBit) + int(bBit), 2)
        total += str(remainder)
    # Return the reversed string.
    return total[::-1]

In [ ]:
# Third, we write a function that converts our binary-fraction string back to a
# decimal-fraction string.
def bin_to_dec(bin):
    """
    Convert the binary-fraction string `bin` into a decimal-string fraction
    using the reverse of the `dec_to_bin` algorithm.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Set the precision sufficiently high.
        ctx.prec = len(bin) + 1
        # Start with 0.
        dec = decimal.Decimal(0)
        # Add each bit as 2^-i, where i is its index.
        for power, bit in enumerate(bin):
            dec += int(bit) / decimal.Decimal(2) ** (power + 1)
        return str(dec)

In [ ]:
# Set the precision. Here, we use 10 bits.
p = 10

# Write out the two fractions.
a = dec_to_bin(1, 10, p)
b = dec_to_bin(2, 10, p)
print("0.1:", a)
print("0.2:", b)

# Add the fractions.
c = add_bin(a, b)
print("sum:", c)

# Convert back to decimal.
d = bin_to_dec(c)
print("dec:", d)

Well, it looks like we certainly have a problem! Can we reproduce the same result as what we got above with `0.1 + 0.2 = 0.3`?

In [ ]:
# Use our functions to calculate the issue for 64 bits.

In [ ]:
# Loop over precisions.
# Get the decimal string.
# Find the first non-nine entry.

In [ ]:
# Redo the calculation above using floats.

In [ ]:
# Loop over some different integer values and find how many bits they use.

In [ ]:
# Use bin_to_dec and dec_to_bin to calculate this error.

In [ ]:
# Calculate the error for 100 hours of operation.

In [ ]:
# Calculate the range gate error.

In [ ]:
# Calculate this rate in gigabytes/second (GB/s).

In [ ]:
# Calculate this updated rate in gigabytes/second (GB/s).

In [ ]:
# Calculate this updated rate in gigabytes/second (GB/s).

In [ ]:
# Write a function that implements this boost.
def boost(E, p, p_b, m_b, stable=False, dec=None):
    """
    Boost a particle with energy `E` and momentum `p` into the frame of
    particle `b` with momentum `p_b` and mass `m_b`. Returns E' and p'.
    If `stable` is `True`, calculate `gamma` as `E_b/m_b`. If `dec`
    is an integer, use the `Decimal` package with this precision.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Use decimal representation and set the precision.
        if type(dec) is int:
            rep = decimal.Decimal
            ctx.prec = dec
        # Use floating-point representation otherwise.
        else:
            rep = float
        # Convert all our values to Decimal or float.
        E, p, p_b, m_b = rep(E), rep(p), rep(p_b), rep(m_b)
        # Calculate the boost energy.
        # Calculate the beta factor.
        # Calculate the gamma factor (either stable or unstable).
        # Perform the boost.
        # Return the result.

In [ ]:
# Mass of the particle whose frame we are boosting (electron mass in GeV).
m_b = 0.000511
# Momentum of the particle whose frame we are boosting.
p_bs = [p_b for p_b in range(1000)]
# Energy and momentum of the particle we are boosting.
E, p = m_b, 0
# Plot the difference between numerically unstable and high precision.
# Plot the difference between stable and high precision.
# Label the plot.